In [27]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

df = pd.read_csv("spotify.csv")

features = [
    "danceability",
    "energy",
    "key",
    "loudness",
    "mode",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
    "duration_ms"
]

X = df[features]
X.head()

,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms
0,0.748,0.916,6,-2.634,1,0.0583,0.1020,0.000000,0.0653,0.518,122.036,194754
1,0.726,0.815,11,-4.969,1,0.0373,0.0724,0.004210,0.3570,0.693,99.972,162600
2,0.675,0.931,1,-3.432,0,0.0742,0.0794,0.000023,0.1100,0.613,124.008,176616
3,0.718,0.930,7,-3.778,1,0.1020,0.0287,0.000009,0.2040,0.277,121.956,169093
4,0.650,0.833,1,-4.672,1,0.0359,0.0803,0.000000,0.0833,0.725,123.976,189052


In [28]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)
X_scaled.head()

,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms
0,0.642049,1.201614,0.173200,1.367123,0.876177,-0.481362,-0.333898,-0.377953,-0.809230,0.031908,0.042927,-0.518874
1,0.490412,0.643317,1.557627,0.585766,0.876177,-0.688642,-0.468670,-0.359177,1.081061,0.782522,-0.777198,-1.056268
2,0.138889,1.284529,-1.211227,1.100090,-1.141322,-0.324422,-0.436799,-0.377849,-0.519562,0.439384,0.116227,-0.822017
3,0.435271,1.279002,0.450085,0.984309,0.876177,-0.050024,-0.667642,-0.377911,0.089582,-1.001795,0.039953,-0.947750
4,-0.033426,0.742815,-1.211227,0.685151,0.876177,-0.702460,-0.432701,-0.377953,-0.692585,0.919777,0.115037,-0.614172


## Standardizing the Features

K-Means is a **distance-based clustering algorithm**, meaning it groups observations based on the Euclidean distance between data points. Because the selected Spotify audio features are measured on different scales, directly applying K-Means would cause features with larger numerical values to dominate the distance calculations.

For example:

- **danceability**, **energy**, and **valence** range from 0 to 1.
- **tempo** is measured in beats per minute (typically 60–200).
- **duration_ms** is measured in hundreds of thousands of milliseconds.

To ensure that each feature contributes equally during clustering, the data is standardized using **StandardScaler**, which transforms every feature to have:

- a mean of **0**
- a standard deviation of **1**

This prevents large-scale features from disproportionately influencing the clustering process.

In [29]:
inertia = []

for k in range(2, 11):
    model = KMeans(
        n_clusters=k,
        random_state=42
    )
    model.fit(X_scaled)
    inertia.append(model.inertia_)

elbow_df = pd.DataFrame({
    "k": range(2, 11),
    "Inertia": inertia
})

fig = px.line(
    elbow_df,
    x="k",
    y="Inertia",
    markers=True,
    title="Elbow Method for Choosing the Number of Clusters"
)

fig.update_layout(
    xaxis_title="Number of Clusters (k)",
    yaxis_title="Inertia",
    template="plotly_white"
)

fig.show()

## Choosing the Number of Clusters – Elbow Method

One of the key challenges in K-Means clustering is selecting an appropriate number of clusters (**k**). The Elbow Method helps identify a suitable value by measuring the **Within-Cluster Sum of Squares (WCSS)**, also known as **inertia**.

As the number of clusters increases, inertia naturally decreases because each cluster contains fewer observations. However, after a certain point, the improvement becomes marginal.

The optimal value of **k** is typically found at the **"elbow"** of the curve, where adding more clusters no longer produces a significant reduction in inertia. This point represents a good balance between model complexity and cluster compactness.bow

In [30]:
scores = []

for k in range(2, 11):
    model = KMeans(
        n_clusters=k,
        random_state=42
    )

    labels = model.fit_predict(X_scaled)

    scores.append(
        silhouette_score(X_scaled, labels)
    )

silhouette_df = pd.DataFrame({
    "k": range(2, 11),
    "Silhouette Score": scores
})

fig = px.line(
    silhouette_df,
    x="k",
    y="Silhouette Score",
    markers=True,
    title="Silhouette Scores for Different Numbers of Clusters"
)

fig.update_layout(
    xaxis_title="Number of Clusters (k)",
    yaxis_title="Silhouette Score",
    template="plotly_white"
)

fig.show()

## Evaluating Cluster Quality – Silhouette Score

While the Elbow Method evaluates how compact the clusters are, it does not indicate how well separated they are. To complement this analysis, the **Silhouette Score** is calculated for different values of **k**.

The Silhouette Score measures how similar each observation is to its own cluster compared with the nearest neighbouring cluster. The score ranges from **-1 to 1**:

- **1** indicates well-separated and highly cohesive clusters.
- **0** suggests overlapping clusters.
- **Negative values** indicate that observations may have been assigned to the wrong cluster.

A higher average Silhouette Score generally indicates better clustering performance.

In [31]:
kmeans = KMeans(
    n_clusters=4,
    random_state=42
)

df["Cluster"] = kmeans.fit_predict(X_scaled)
df["Cluster"] = df["Cluster"].astype("category")

In [32]:
pca = PCA(n_components=2)

components = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(
    components,
    columns=["PC1", "PC2"]
)

pca_df["Cluster"] = df["Cluster"]

fig = px.scatter(
    pca_df,
    x="PC1",
    y="PC2",
    color="Cluster",
    title="K-Means Clusters (PCA Projection)",
    opacity=0.8,
    height=700
)

fig.update_layout(
    template="plotly_white"
)

fig.show()

In [33]:
cluster_summary = (
    df
    .groupby("Cluster")[features]
    .mean()
    .round(3)
)

cluster_summary

/var/folders/89/f_fc311s7v977_m_np2qc56c0000gn/T/ipykernel_53300/3313565994.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("Cluster")[features]


,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms
Cluster,,,,,,,,,,,,
0,0.558,0.800,5.052,-5.290,0.615,0.085,0.073,0.022,0.236,0.422,132.305,226715.147
1,0.661,0.785,5.372,-6.950,0.561,0.072,0.072,0.737,0.191,0.389,125.170,251498.791
2,0.606,0.436,5.207,-10.394,0.609,0.088,0.503,0.086,0.158,0.398,113.069,226233.456
3,0.751,0.701,5.700,-6.414,0.510,0.138,0.151,0.011,0.165,0.649,113.855,220022.652


In [34]:
fig = px.box(
    df,
    x="Cluster",
    y="track_popularity",
    color="Cluster",
    title="Track Popularity by Cluster"
)

fig.update_layout(
    template="plotly_white",
    showlegend=False
)

fig.show()

In [35]:
cluster_counts = (
    df["Cluster"]
    .value_counts()
    .sort_index()
    .reset_index()
)

cluster_counts.columns = ["Cluster", "Songs"]

fig = px.bar(
    cluster_counts,
    x="Cluster",
    y="Songs",
    color="Cluster",
    title="Number of Songs in Each Cluster",
    text="Songs"
)

fig.update_layout(
    template="plotly_white",
    showlegend=False
)

fig.show()